In [ ]:
import pandas as pd
from scipy import stats
from config import FEATURED_PATH

In [ ]:
df = pd.read_csv(FEATURED_PATH)

In [ ]:
# Correlation: quality_score vs discrepancy_count (expect negative)
corr, pval = stats.pearsonr(df["quality_score"], df["discrepancy_count"])
print(f"Correlation quality_score vs discrepancy_count: r={corr:.3f}, p={pval:.3f}")
print("(n=41 -> treat p-value as directional context only, not a powered inferential result)\n")

## Correlation: quality score vs. discrepancy count

**r = -0.470, p = 0.002** (n = 41)

There's a moderate negative correlation between a facility's overall data
quality score and the number of discrepancies found during the visit — as
the quality score goes up, the discrepancy count tends to go down. This is
the direction we'd expect if the quality score is actually measuring what
it's supposed to measure, so it's a useful internal consistency check on
the checklist itself.

**Note on interpretation:** with n = 41, this p-value is reported as
directional context, not as evidence of a statistically powered effect.
The result supports treating `quality_score` as a reasonably reliable
summary measure for this dataset, but shouldn't be read as a generalizable
finding beyond it.

In [ ]:
# Group comparison by sub-county (descriptive, not inferential)
summary = df.groupby("sub_county").agg(
    avg_quality_score=("quality_score", "mean"),
    total_discrepancies=("discrepancy_count", "sum"),
    late_submissions=("reporting_status", lambda x: (x != "On time").sum()),
    n_visits=("facility_name", "count")
).round(2).sort_values("avg_quality_score")
 
print("Descriptive summary by sub-county (n too small for inferential claims):")
print(summary)
 

## Descriptive summary by sub-county

*(n too small for inferential claims — reported descriptively only)*

| Sub-county | Avg quality score | Total discrepancies | Late submissions | Visits |
|---|---|---|---|---|
| Turbo | 2.17 | 62 | 1 | 6 |
| Kapseret | 3.33 | 52 | 3 | 6 |
| Kesses | 3.33 | 35 | 3 | 6 |
| Soy | 3.38 | 71 | 2 | 8 |
| Moiben | 3.50 | 24 | 5 | 6 |
| Ainabkoi | 3.67 | 38 | 2 | 9 |

**Turbo stands out.** It has the lowest average data quality score (2.17) of
all six sub-counties, yet only 1 of its 6 visits was flagged as a late
submission — the second-fewest late submissions in the dataset. This is a
meaningful finding: reporting *timeliness* alone would rank Turbo as one of
the better-performing sub-counties, while the quality checklist reveals real
underlying data problems that timeliness data would completely miss. This is
the core justification for pairing supervision-checklist data with routine
reporting metrics rather than relying on timeliness alone.

**Moiben**, by contrast, has the most late submissions (5 of 6 visits) but a
comparatively strong average quality score (3.50) — suggesting its reporting
delays aren't linked to poor data quality once the report does arrive.

**Caveat:** with 6-9 visits per sub-county, these are descriptive patterns
worth flagging for follow-up, not statistically confirmed differences.

In [ ]:
# Missingness validation: delay_reason should be fully explained by report_submitted
mismatch = df[(df["report_submitted"].str.lower() == "yes") & (df["delay_reason"] != "N/A - submitted on time")]
print(f"\nRows where report_submitted=Yes but delay_reason wasn't auto-filled: {len(mismatch)} (should be 0)")
 
still_missing = df[(df["report_submitted"].str.lower() == "no") & (df["delay_reason"].isna())]
print(f"Genuine missing delay_reason among late submissions: {len(still_missing)} (real gap, not structural)")
 

## Missingness validation 
confirmed structural-missingness handling worked correctly, with the one real gap flagged rather than hidden

In [ ]:
q1 = df["discrepancy_count"].quantile(0.25)
q3 = df["discrepancy_count"].quantile(0.75)
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr

outliers = df[(df["discrepancy_count"] < lower) | (df["discrepancy_count"] > upper)]
print(f"IQR bounds: [{lower:.1f}, {upper:.1f}]")
outliers[["sub_county", "facility_name", "discrepancy_count"]]

## Outlier check: discrepancy count

Using the IQR method (Q1 - 1.5×IQR to Q3 + 1.5×IQR), the upper bound for
`discrepancy_count` is **18.5**. Three visits exceed it:

| Sub-county | Facility | Discrepancies |
|---|---|---|
| Soy | Ziwa Sub-County Hospital | 43 |
| Turbo | Huruma Sub-County Hospital | 43 |
| Kesses | Kesses Sub-County Hospital | 19 |

**This resolves the Soy concern from the sub-county summary.** Soy's high
total (71 discrepancies across 8 visits) is driven almost entirely by a
single outlier visit to Ziwa Sub-County Hospital (43 discrepancies) — not a
broad pattern across all of Soy's facilities. Ziwa itself is otherwise a
strong performer (three separate visits scored 5/5 with 0 discrepancies),
so this looks like one unusually bad visit rather than a systemic issue at
that facility.

**Turbo's outlier is more concerning.** Huruma Sub-County Hospital's 43
discrepancies contributes to Turbo's already-low average quality score
(2.17) identified earlier — this isn't an isolated spike pulling down an
otherwise-fine sub-county average, it reinforces the pattern already seen.

These three flagged visits are kept in the dataset (not removed) since
they're plausible real supervision findings, not data entry errors — but
they're worth highlighting individually if this dashboard is ever reviewed,
since they're the specific incidents driving the sub-county-level numbers.